# DBSCAN Cluster Preview For Original Events

This notebook loads the original event locations, runs DBSCAN on epicentral coordinates, and visualizes possible spatial clusters before running HypoDD.

DBSCAN here uses a haversine distance, so `eps_km` is interpreted in kilometers. This is only a spatial preview; it does not use picks, stations, or differential-time links.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.cluster import DBSCAN

try:
    from obspy import read_events, read_inventory
except ImportError:
    read_events = None
    read_inventory = None

EARTH_RADIUS_KM = 6371.0088
plt.rcParams["figure.figsize"] = (8, 7)
plt.rcParams["axes.grid"] = True

## Input Files And DBSCAN Parameters

Use `events.xml` if you already converted your CSV files to QuakeML. If not, set `EVENT_XML = None` and use `EVENT_CSV`.

In [ ]:
EVENT_XML = Path("events.xml")
STATION_XML = Path("stations.xml")

# CSV fallback. Used only if EVENT_XML is None or does not exist.
EVENT_CSV = Path("data/starting-event-Malyn.csv")

# Try several eps values to see when bridge events start merging clusters.
EPS_VALUES_KM = [1.5, 2.0, 2.5, 3.0, 4.0, 5.0]

# DBSCAN min_samples is the minimum number of points in an eps-neighborhood
# for a point to be a core point. It is not exactly minimum final cluster size.
MIN_SAMPLES = 8

# Choose one eps for the larger final map below.
SELECTED_EPS_KM = 3.0

In [ ]:
def catalog_to_dataframe(path):
    if read_events is None:
        raise ImportError("ObsPy is required to read QuakeML. Install obspy or use the CSV fallback.")
    catalog = read_events(str(path))
    rows = []
    for index, event in enumerate(catalog):
        origin = event.preferred_origin() or (event.origins[0] if event.origins else None)
        if origin is None or origin.latitude is None or origin.longitude is None:
            continue
        magnitude = event.preferred_magnitude()
        rows.append(
            {
                "event_id": str(event.resource_id),
                "event_index": index,
                "time": origin.time.datetime if origin.time else pd.NaT,
                "latitude": float(origin.latitude),
                "longitude": float(origin.longitude),
                "depth_km": float(origin.depth or 0.0) / 1000.0,
                "magnitude": float(magnitude.mag) if magnitude and magnitude.mag is not None else np.nan,
            }
        )
    return pd.DataFrame(rows)


def csv_to_dataframe(path):
    df = pd.read_csv(path)
    lower_to_original = {column.lower(): column for column in df.columns}
    lat_col = lower_to_original.get("latitude") or lower_to_original.get("lat")
    lon_col = lower_to_original.get("longitude") or lower_to_original.get("lon")
    if lat_col is None or lon_col is None:
        raise ValueError(f"Could not find latitude/longitude columns in {path}. Columns: {list(df.columns)}")
    event_id_col = lower_to_original.get("id") or lower_to_original.get("eventid") or lower_to_original.get("event_id")
    depth_col = lower_to_original.get("depth") or lower_to_original.get("depth_km")
    time_col = lower_to_original.get("isotime") or lower_to_original.get("time")
    out = pd.DataFrame(
        {
            "event_id": df[event_id_col].astype(str) if event_id_col else df.index.astype(str),
            "event_index": np.arange(len(df)),
            "time": pd.to_datetime(df[time_col], errors="coerce") if time_col else pd.NaT,
            "latitude": pd.to_numeric(df[lat_col], errors="coerce"),
            "longitude": pd.to_numeric(df[lon_col], errors="coerce"),
            "depth_km": pd.to_numeric(df[depth_col], errors="coerce") if depth_col else np.nan,
        }
    )
    return out.dropna(subset=["latitude", "longitude"]).reset_index(drop=True)


def inventory_to_dataframe(path):
    if path is None or not Path(path).exists() or read_inventory is None:
        return pd.DataFrame(columns=["station_id", "latitude", "longitude"])
    inventory = read_inventory(str(path))
    rows = []
    for network in inventory:
        for station in network:
            rows.append(
                {
                    "station_id": f"{network.code}.{station.code}",
                    "latitude": float(station.latitude),
                    "longitude": float(station.longitude),
                }
            )
    return pd.DataFrame(rows)

In [ ]:
if EVENT_XML is not None and Path(EVENT_XML).exists():
    events = catalog_to_dataframe(EVENT_XML)
    print(f"Loaded {len(events)} events from {EVENT_XML}")
else:
    events = csv_to_dataframe(EVENT_CSV)
    print(f"Loaded {len(events)} events from {EVENT_CSV}")

stations = inventory_to_dataframe(STATION_XML)
print(f"Loaded {len(stations)} stations from {STATION_XML if STATION_XML.exists() else 'none'}")

events.head()

In [ ]:
def run_dbscan(events_df, eps_km, min_samples):
    coords_rad = np.radians(events_df[["latitude", "longitude"]].to_numpy())
    model = DBSCAN(
        eps=eps_km / EARTH_RADIUS_KM,
        min_samples=min_samples,
        metric="haversine",
    )
    labels = model.fit_predict(coords_rad)
    clustered = events_df.copy()
    clustered["dbscan_cluster"] = labels
    return clustered


def cluster_summary(clustered):
    counts = clustered["dbscan_cluster"].value_counts().sort_index()
    rows = []
    for cluster_id, count in counts.items():
        rows.append(
            {
                "cluster": cluster_id,
                "event_count": int(count),
                "is_noise": cluster_id == -1,
            }
        )
    return pd.DataFrame(rows)


summaries = []
clustered_by_eps = {}
for eps_km in EPS_VALUES_KM:
    clustered = run_dbscan(events, eps_km=eps_km, min_samples=MIN_SAMPLES)
    clustered_by_eps[eps_km] = clustered
    counts = clustered["dbscan_cluster"].value_counts()
    summaries.append(
        {
            "eps_km": eps_km,
            "min_samples": MIN_SAMPLES,
            "clusters": int((counts.index != -1).sum()),
            "noise_events": int(counts.get(-1, 0)),
            "largest_cluster": int(counts[counts.index != -1].max()) if (counts.index != -1).any() else 0,
        }
    )

pd.DataFrame(summaries)

In [ ]:
def plot_dbscan_map(clustered, stations=None, title=None, ax=None):
    if ax is None:
        fig, ax = plt.subplots(figsize=(8, 7))
    clusters = sorted(clustered["dbscan_cluster"].unique())
    non_noise = [cluster for cluster in clusters if cluster != -1]
    cmap = plt.get_cmap("tab20")

    noise = clustered[clustered["dbscan_cluster"] == -1]
    if len(noise):
        ax.scatter(
            noise["longitude"],
            noise["latitude"],
            marker="x",
            s=16,
            c="0.65",
            linewidths=0.8,
            label=f"Noise ({len(noise)})",
        )

    for index, cluster_id in enumerate(non_noise):
        subset = clustered[clustered["dbscan_cluster"] == cluster_id]
        ax.scatter(
            subset["longitude"],
            subset["latitude"],
            marker="o",
            s=14,
            alpha=0.75,
            color=cmap(index % cmap.N),
            label=f"Cluster {cluster_id} ({len(subset)})",
        )

    if stations is not None and len(stations):
        ax.scatter(
            stations["longitude"],
            stations["latitude"],
            marker="^",
            s=45,
            c="black",
            edgecolors="white",
            linewidths=0.5,
            label="Stations",
            zorder=5,
        )

    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
    ax.set_title(title or "DBSCAN event clusters")
    ax.set_aspect("equal", adjustable="box")
    ax.legend(loc="best", fontsize=7, markerscale=1.2)
    return ax

In [ ]:
selected = run_dbscan(events, eps_km=SELECTED_EPS_KM, min_samples=MIN_SAMPLES)
display(cluster_summary(selected))

ax = plot_dbscan_map(
    selected,
    stations=stations,
    title=f"DBSCAN preview: eps={SELECTED_EPS_KM} km, min_samples={MIN_SAMPLES}",
)
plt.show()

## Compare Several `eps_km` Values

This is useful for finding the distance threshold where two visually separate event clouds start to merge through bridge events.

In [ ]:
ncols = 3
nrows = int(np.ceil(len(EPS_VALUES_KM) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 4.5 * nrows), squeeze=False)

for ax, eps_km in zip(axes.ravel(), EPS_VALUES_KM):
    clustered = clustered_by_eps[eps_km]
    counts = clustered["dbscan_cluster"].value_counts()
    n_clusters = int((counts.index != -1).sum())
    n_noise = int(counts.get(-1, 0))
    plot_dbscan_map(
        clustered,
        stations=stations,
        title=f"eps={eps_km} km | clusters={n_clusters} | noise={n_noise}",
        ax=ax,
    )
    ax.legend().remove()

for ax in axes.ravel()[len(EPS_VALUES_KM):]:
    ax.axis("off")

plt.tight_layout()
plt.show()

## Save The Preview Labels

This writes a CSV with the selected DBSCAN label for each event. Cluster `-1` means DBSCAN noise/unclustered.

In [ ]:
OUTPUT_CSV = Path(f"dbscan_preview_eps_{SELECTED_EPS_KM:g}km_min_{MIN_SAMPLES}.csv")
selected.to_csv(OUTPUT_CSV, index=False)
OUTPUT_CSV